# 09 · Inspect the project database

Run the first Airflow DAG before querying the tables.

## Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd
from sqlalchemy import text

sys.path.insert(0, str(Path('../src').resolve()))
from property_ml.settings import engine

db = engine()


## Batches

In [2]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM raw.batches ORDER BY group_number, batch_number'), conn))


,group_number,batch_number,run_id,received_at,row_count,status,received_columns,valid_rows,rejected_rows,duplicate_rows,quality
0,4,0,manual__2026-11-23T16:17:08.395978+00:00,2026-11-23 16:16:57.136017+00:00,73784,cleaned,"[acre_lot, bath, bed, brokered_by, city, house...",73784,0,0,"{'missing': {'bed': 0, 'bath': 0, 'city': 0, '..."


## Raw and clean counts

In [3]:
with db.connect() as conn:
    for table in ['raw.properties', 'clean.properties', 'audit.inference_events']:
        print(table, conn.execute(text(f'SELECT count(*) FROM {table}')).scalar())


raw.properties 73784
clean.properties 73784
audit.inference_events 0


## Decisions and model history

In [7]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT * FROM audit.batch_decisions ORDER BY batch_number'), conn))
    display(pd.read_sql_query(text('SELECT * FROM audit.training_events ORDER BY id'), conn))


,group_number,batch_number,created_at,valid_rows,pending_rows,decision,reason,model_version,candidate_mae,champion_mae,promotion,run_id,metrics
0,4,0,2026-11-23 16:17:32.839459+00:00,73784,73784,train,initial baseline,1,44278.238084,None,initial model,manual__2026-11-23T16:17:08.395978+00:00,"{'psi': {}, 'contribution': None, 'new_fractio..."


,id,group_number,batch_number,created_at,mlflow_run_id,model_version,candidate_mae,champion_mae,promoted,reason
0,1,4,0,2026-11-23 16:18:12.660066+00:00,1211bc18d93d49979c3de5f76bcf5d14,1,44278.238084,None,True,initial model


## Sample predictions

In [6]:
with db.connect() as conn:
    display(pd.read_sql_query(text('SELECT created_at, prediction, model_version, pod, status, error FROM audit.inference_events ORDER BY id DESC LIMIT 10'), conn))
db.dispose()


,created_at,prediction,model_version,pod,status,error
0,2026-11-23 16:34:51.770823+00:00,239949.4375,1,property-api-dd8476699-hwz48,success,None
